# T12 후두 자질 분석 (Willett et al. 2023)**핵심 질문**: 영어의 후두 대립은 말소리 운동 계획에서 `[voice]`(성대 진동)로 표상되는가,`[spread glottis]`(성문 벌림·기식)로 표상되는가.| 시험 | 측정 | `[voice]` 예측 | `[spread glottis]` 예측 ||---|---|---|---|| T1 위치 | 어두 vs 어중 vs 어말 p/b 분리도 | 어중 최강 | 어두 최강 || T2 유표성 | 어느 계열이 무표 기준선에서 먼가 | b d g | p t k || T3 양식 | 무성 발화에서 후두 정보 보존 | 소멸 | 부분 보존 |구성은 VocalMind 노트북(`colab/notebook.ipynb`)과 같다:경로 → 데이터 확인 → 인벤토리 → QC → 음성 주석 → Stage 1 → Stage 2 → 요약.모든 셀은 결과를 `RESULT_DIR`에 캐시하고 재실행 시 재사용한다.분석 코드는 전부 `t12_pipeline.py`에 있고 이 노트북은 그것을 `import`해서 호출만 한다.

In [ ]:
# =====================================================================
# 1. 경로 설정 — Colab이면 Drive 마운트, 로컬이면 t12/ 하위 경로
# =====================================================================
LOCAL = True   # Colab에서 돌릴 때는 False 로 바꾼다

import sys
from pathlib import Path

if LOCAL:
    CODE_DIR = Path(
        "/Users/hojaechoi/Documents/research/speech-bci-confusion/t12/code"
    )
else:
    from google.colab import drive

    drive.mount("/content/drive")
    CODE_DIR = Path("/content/drive/MyDrive/T12_Project/03_code")
    if not (CODE_DIR / "t12_pipeline.py").exists():
        # Drive에 모듈이 없으면 GitHub 레포(g00dnew/Colab)에서 받아 온다
        import urllib.request
        CODE_DIR = Path("/content/t12_code")
        CODE_DIR.mkdir(exist_ok=True)
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/g00dnew/Colab/main/t12/t12_pipeline.py",
            CODE_DIR / "t12_pipeline.py",
        )
        print("t12_pipeline.py 를 GitHub에서 받았습니다:", CODE_DIR)

if not CODE_DIR.is_dir():
    raise FileNotFoundError(
        f"코드 폴더를 찾을 수 없습니다: {CODE_DIR}\n"
        "t12_pipeline.py 를 이 폴더에 복사하세요 "
        "(Colab이면 Drive의 T12_Project/03_code/)."
    )

sys.path.insert(0, str(CODE_DIR))

import importlib

import numpy as np
import pandas as pd

import t12_pipeline as tp

importlib.reload(tp)

paths = tp.get_paths(local=LOCAL)

print("연구 폴더 연결 완료")
print("  ROOT       :", paths["ROOT"])
print("  ARCHIVE_DIR:", paths["ARCHIVE_DIR"])
print("  DATA_DIR   :", paths["DATA_DIR"])
print("  ANNOT_DIR  :", paths["ANNOT_DIR"])
print("  RESULT_DIR :", paths["RESULT_DIR"])
print("  ALIGN_DIR  :", paths["ALIGN_DIR"])
print()
print("어레이 채널 집합(readme 지도 기준):")
for name in ("6v", "6v_sup", "6v_inf", "44", "44_sup", "44_inf", "all"):
    ch = tp.ARRAYS[name]
    print(f"  {name:<7} 채널 {ch.size:>3}개  -> 특징 열 {tp.array_columns(name).size}개")

In [ ]:
# =====================================================================
# 2. 데이터 확인 — Dryad는 봇을 차단하므로 여기서 내려받지 않는다
# =====================================================================
# 브라우저로 받아 DATA_DIR(Colab이면 Drive의 01_data)에 둔 tar.gz를 확인하고
# 압축을 푼다. 이미 풀려 있으면 gzip 재검사(수 분)를 생략한다.

WANT = ["tuningTasks.tar.gz", "competitionData.tar.gz"]
VERIFY_GZIP = False   # True로 두면 아카이브 전체를 다시 읽는다(수 분 소요)

already = {
    name: (paths["DATA_DIR"] / tp.ARCHIVES[name]["member"]).is_dir()
    for name in WANT
}
print("압축 해제 상태:", already)

need_check = [n for n in WANT if not already[n]]
if need_check:
    status = tp.check_archives(paths, names=need_check, verify_gzip=VERIFY_GZIP)
    display(status)
    for name in need_check:
        tp.extract_archive(paths, name)
else:
    print("두 데이터셋 모두 이미 풀려 있습니다. 압축 해제 생략.")
    if VERIFY_GZIP:
        display(tp.check_archives(paths, names=WANT, verify_gzip=True))

print()
print("tuningTasks:", sorted(p.name for p in paths["TUNING_DIR"].glob("*.mat")))
for part in ("train", "test", "competitionHoldOut"):
    d = paths["COMP_DIR"] / part
    print(f"competitionData/{part}:", len(list(d.glob('*.mat'))) if d.is_dir() else "없음")

print()
print("수동 다운로드가 필요하면:", tp.DRYAD_URL)

In [ ]:
# =====================================================================
# 3. 세션 인벤토리
# =====================================================================
# competitionData: 세션 x 파티션별 시행 수, 블록 수, 양식 라벨, 총 길이
# tuningTasks   : 파일별 cue, cue별 시행 수

tuning_inv = tp.tuning_inventory(paths)
if not tuning_inv.empty:
    display(tuning_inv[["file", "task", "modality", "n_trials", "n_blocks",
                        "n_cues", "trials_per_cue_min", "go_bins_mean",
                        "total_sec", "trial_states"]])

cue_counts = tp.tuning_cue_counts(paths)
if not cue_counts.empty:
    print("\nphonemes 과제 cue -> ARPABET 매핑 (앞 8개):")
    display(cue_counts.head(8))
    print("세션별 cue당 시행 수:")
    display(cue_counts.pivot_table(index="session", values="n_trials",
                                   aggfunc=["min", "max", "count"]))

comp_inv = tp.competition_inventory(paths)
if not comp_inv.empty:
    display(comp_inv)
    print("\n파티션 x 양식 집계:")
    display(comp_inv.groupby(["partition", "modality"]).agg(
        n_sessions=("session", "nunique"),
        n_trials=("n_trials", "sum"),
        total_sec=("total_sec", "sum"),
    ))
    nonvocal = sorted(comp_inv.loc[comp_inv["modality"] == "nonvocal", "session"].unique())
    print("\nnonvocal 세션:", nonvocal)
    assert set(nonvocal) <= tp.NONVOCAL_SESSIONS, "양식 라벨이 저장소 라벨과 다릅니다"

In [ ]:
# =====================================================================
# 4. QC — 모양, NaN/Inf, 블록 z-score 전후 채널 통계, 시행 길이 분포
# =====================================================================
# competitionData는 세션 48~63개를 모두 읽으므로 몇 분 걸린다(결과는 캐시된다).

tuning_q = tp.tuning_qc(paths)
if not tuning_q.empty:
    display(tuning_q)

MAX_QC_SESSIONS = None   # 빠르게 보려면 숫자를 넣는다 (예: 6)
comp_q = tp.competition_qc(paths, max_sessions=MAX_QC_SESSIONS)
if not comp_q.empty:
    display(comp_q.head(12))
    print("\nz-score 후 채널 평균의 절대 최대값(0에 가까워야 한다):",
          round(float(comp_q["z_chan_mean_absmax"].max()), 6))
    print("z-score 후 채널 표준편차 중앙값(1에 가까워야 한다):",
          round(float(comp_q["z_chan_std_med"].median()), 4))
    print("NaN 총합:", int(comp_q["n_nan"].sum()),
          "| Inf 총합:", int(comp_q["n_inf"].sum()),
          "| 죽은 채널 최대:", int(comp_q["n_dead_chan"].max()))
    assert comp_q["n_nan"].sum() == 0 and comp_q["n_inf"].sum() == 0, "NaN/Inf 발견"

In [ ]:
# =====================================================================
# 5. 음성 자질 주석 (02_annotations)
# =====================================================================
# ARPABET 자음 x (유성성, 조음위치, 조음방법, 비음, 기식) + 최소대립쌍 라벨.
# verified=False, 사전등록 전 확정 금지.

features = tp.build_phonetic_feature_table(paths)
display(features)

pairs = pd.read_csv(paths["ANNOT_DIR"] / "minimal_pairs_v1.csv")
display(pairs)

print("후두 최소대립쌍:", [f"{a}/{b}" for a, b in tp.LARYNGEAL_PAIRS])
print("조음위치 통제쌍:", [f"{a}/{b}" for a, b in tp.PLACE_CONTROL_PAIRS])
print("유표성 기준선 음소 집합:", tp.PAIRED_OBSTRUENTS)

assert len(features) == 24
assert features["arpabet"].is_unique
assert (~features["verified"]).all(), "verified=True가 섞였습니다 (사전등록 전 확정 금지)"
for a, b in tp.LARYNGEAL_PAIRS:
    row_a = features.set_index("arpabet").loc[a]
    row_b = features.set_index("arpabet").loc[b]
    assert row_a["place"] == row_b["place"], f"{a}/{b} 조음위치 불일치"
    assert row_a["voicing"] != row_b["voicing"], f"{a}/{b} 유성성 동일"
print("\n자질표 정합성 점검 통과")

In [ ]:
# =====================================================================
# 6. Stage 1 — 정렬 없이 되는 분석 (tuningTasks 고립 음소)
# =====================================================================
# 시행별 특징 = 창 내 tx1/spikePow 시간평균, 블록별 z-score.
# 창 두 종류:
#   go_*    : go 신호 기준. 음성 데이터가 없는 세션에서도 쓸 수 있다.
#   onset_* : audioEnvelope로 추정한 발화 onset 기준 고정 길이.
#             모든 쌍에 같은 길이를 쓰므로 조음방법별 지속시간 차이가
#             신호량 차이로 새는 것을 막는다(마찰음이 길어서 유리한 문제).
# 어레이: 6v_sup vs 6v_inf 가 H_sampling(후두 정보가 dorsal에 있다)의 실제 시험이다.
#         area 44 는 거의 우연수준이므로 6v 분할이 본 비교다.

QUICK = False                       # True면 순열 50회로 빠르게 점검
N_PERM = 50 if QUICK else 500
ARRAYS_S1 = ("6v", "6v_sup", "6v_inf", "44", "all")
WINDOWS_S1 = ("go_all", "go_0_500ms", "onset_0_300ms", "onset_0_500ms")

feats = tp.load_stage1_features(paths)      # onset 창까지 함께 만든다
print("시행", feats["go_all"].shape[0], "x 특징", feats["go_all"].shape[1])
print("사용 가능한 창:", feats["windows"])
if "onset_bins" in feats:
    ob = pd.Series(feats["onset_bins"])
    print(f"발화 onset(go 기준): 중앙 {int(ob.median())} bin "
          f"({int(ob.median()) * 20} ms), p10 {int(ob.quantile(0.1))}, "
          f"p90 {int(ob.quantile(0.9))}")
    print("주의: audioEnvelope가 있는 세션은 04.26뿐이다. "
          "04.21은 onset=중앙값으로 대체되므로 onset 분석의 1차 근거는 04.26 단독이다.")
assert feats["go_all"].shape[1] == 512

# 6-1. 39음소 분류 = 전건전성 점검 (저자 보고대로 우연수준을 크게 넘어야 한다)
p39 = tp.stage1_phoneme39(paths, feats=feats, window="go_all", arrays=ARRAYS_S1)
display(p39)

# 6-2. 창별/어레이별 무순열 스캔 (지속시간 교란 점검 포함)
scan_path = paths["RESULT_DIR"] / "stage1_onset_window_scan.csv"
if scan_path.exists():
    scan = pd.read_csv(scan_path)
    print("\n창 x 어레이 x 대립 유형 평균 정확도:")
    display(scan[scan["session_set"] == "pooled"]
            .pivot_table(index=["window", "array"], columns="contrast",
                         values="acc").round(4))
    print("\n후두쌍: 창 x 조음방법 (area 6v, 04.26 단독 = 진짜 onset 기준):")
    s26 = scan[(scan["session_set"] == "audio_0426") & (scan["array"] == "6v")
               & (scan["contrast"] == "laryngeal")]
    display(s26.pivot_table(index="window", columns="manner", values="acc").round(4))
    display(s26.pivot_table(index=["manner", "pair"], columns="window",
                            values="acc").round(4))

# 6-3. 최소대립쌍 순열검정
s1 = tp.stage1_run(paths, feats=feats, windows=WINDOWS_S1,
                   arrays=ARRAYS_S1, n_perm=N_PERM)
print("\n어레이 x 대립 유형 (go 구간 전체):")
display(s1[s1["window"] == "go_all"]
        .pivot_table(index="array", columns="contrast", values="acc").round(4))
print("\n조음방법 집계 (후두쌍만):")
man = tp.manner_aggregate(s1, group_cols=("window", "array"))
man.to_csv(paths["RESULT_DIR"] / "stage1_manner_aggregate.csv", index=False)
display(man[man["array"] == "6v"].round(4))

# 6-4. T2 유표성 (고립 음소)
mark = tp.stage1_markedness(paths, feats=feats, window="go_all", arrays=ARRAYS_S1)
tests = pd.DataFrame([tp.markedness_contrast_test(mark, array=a)
                      for a in ARRAYS_S1 if a in set(mark["array"])])
tests.to_csv(paths["RESULT_DIR"] / "stage1_markedness_test.csv", index=False)
display(tests)

# 6-5. 그림
tp.fig_stage1_pairs(s1, paths["FIG_DIR"] / "stage1_pair_decoding.png", window="go_all")
tp.fig_markedness(mark, paths["FIG_DIR"] / "stage1_markedness.png", array="6v")

In [ ]:
# =====================================================================
# 7. Stage 2 — 강제 정렬 결과를 쓰는 분석 (T1 위치, T2 유표성, T3 양식, 자질 전달량)
# =====================================================================
# t12/results/alignments/<session>_<partition>.csv 를 하나씩 집어넣는 구조로,
# 그 시점에 존재하는 CSV만 모아 쓰고 어떤 세션이 들어갔는지 보고한다.
#
# 창 선택(window_mode):
#   'seg'        seg_start_bin/seg_end_bin. 시행을 빈틈 없이 분할(중앙 160 ms). 기본.
#   'rnn_center' rnn_center_start/end. 점유 core step만(중앙 80 ms). 민감도.
#   'legacy'     start_bin/end_bin = RNN 수용영역(32 bin, 설계상 겹침). 쓰지 말 것.
# load_alignments가 인접 구간 겹침 비율을 매번 출력한다.
#
# held_out = (partition=='test') or (input_layer_from != session)
#   사전학습 RNN의 학습 설정에 없던 세션(06.23, 07.29, 08.18, 08.23, 08.25)은
#   두 파티션 모두 held out이고, 학습된 19개 세션은 test만 held out이다
#   (학습된 세션의 train 정렬 PER은 ~0.1%로 암기 수준).
#
# 반드시 사전등록 문서에 적을 것:
#   1. CTC는 peaky하므로 음소 경계는 ~80 ms 수준의 근사다.
#   2. S/Z, SH/ZH는 영어에서 어두에 거의 없으므로 T1 어두 조건에 기여하지 못한다.
#      T1은 P/B, T/D, K/G, F/V, TH/DH, CH/JH 6쌍에 의존한다.
#   3. 미국영어 설탄음화로 모음 사이 T/D는 둘 다 설탄음이 된다. 어중 T/D는
#      비모음간 맥락으로 제한한 행을 함께 봐야 한다(drop_intervocalic_flap).
#   4. 위치별 시행 수가 크게 다르므로 match_n_across_positions=True 로
#      같은 쌍이 세 위치에서 동일한 n으로 학습되게 맞춘다.
#   5. 판정이 걸린 양은 '위치 x 대립유형 상호작용'이므로 격차와 DoD에 CI를 붙인다.

WINDOW_MODE = "seg"
SCORE_MIN = 0.5
N_PERM = 100
ARRAYS_S2 = tp.STAGE2_ARRAYS          # ('6v', '6v_sup', '6v_inf')
NONVOCAL = sorted(tp.NONVOCAL_SESSIONS)
VOCAL_PRIMARY = ["t12.2022.07.29"]    # 미학습 + 입력층 차용 = nonvocal과 품질 일치
VOCAL_DATEMATCH = ["t12.2022.08.11", "t12.2022.08.13",
                   "t12.2022.06.21", "t12.2022.06.28"]

per_table = tp.session_per_table(paths["ALIGN_DIR"])
per_table.to_csv(paths["RESULT_DIR"] / "alignment_per.csv", index=False)

align_raw = tp.load_alignments(paths["ALIGN_DIR"], window_mode=WINDOW_MODE)
align, score_report = tp.apply_score_filter(align_raw, score_min=SCORE_MIN)
X, X_pre, meta = tp.stage2_segment_features(paths, align)

print("\n세션:", sorted(meta["session"].unique()))
print("양식:", meta["modality"].value_counts().to_dict())
print("파티션:", meta["partition"].value_counts().to_dict())
print("held_out 구간:", int(meta["held_out"].sum()), "/", len(meta))
assert X.shape == (len(meta), 512), X.shape
assert np.isfinite(X).all()
display(score_report)

# 자음 x 위치 구간 수 (S/Z, SH/ZH 어두 결손을 눈으로 확인)
nbp = (meta[meta["phoneme"].isin(tp.CONSONANTS)]
       .groupby(["phoneme", "position_in_word"]).size().unstack(fill_value=0))
nbp.to_csv(paths["RESULT_DIR"] / "stage2_n_by_position.csv")
display(nbp)

# ---- T1: 위치 (통제 3종) --------------------------------------------
T1_RUNS = [("all_pairs_pooled", dict(partitions=None, drop_intervocalic_flap=False), N_PERM),
           ("all_pairs_heldout", dict(partitions="held_out", drop_intervocalic_flap=False), 0),
           ("flapctrl_pooled", dict(partitions=None, drop_intervocalic_flap=True), 0)]
t1, inter = {}, []
for label, kw, nperm in T1_RUNS:
    d = tp.stage2_position_decode(paths, X, meta, arrays=("6v",), n_perm=nperm,
                                  n_repeats=10, tag=f"_{label}", force=True,
                                  max_per_class=300,
                                  match_n_across_positions=True, **kw)
    if not len(d):
        print(f"[T1 {label}] 결과 없음")
        continue
    t1[label] = d
    print(f"\n[T1 {label}] 위치 x 대립 유형:")
    display(d.pivot_table(index="position", columns="contrast", values="acc").round(4))
    display(d.pivot_table(index=["contrast", "manner", "pair"], columns="position",
                          values="acc").round(4))
    for excl, ename in [((), "none"), (("T/D",), "T/D")]:
        it = tp.position_contrast_interaction(d, array="6v", exclude_pairs=excl)
        it["run"], it["excluded"] = label, ename
        inter.append(it)
if inter:
    inter_df = pd.concat(inter, ignore_index=True)
    inter_df.to_csv(paths["RESULT_DIR"] / "stage2_T1_interaction.csv", index=False)
    print("\n[T1] 위치별 격차 + 차이의 차이 (부트스트랩 95% CI):")
    display(inter_df)

dsp = tp.stage2_position_decode(paths, X, meta, arrays=("6v_sup", "6v_inf"),
                                n_perm=0, n_repeats=10, tag="_arraysplit",
                                force=True, max_per_class=300,
                                match_n_across_positions=True)
if len(dsp):
    print("\n[T1] 어레이 분할:")
    display(dsp.pivot_table(index=["array", "position"], columns="contrast",
                            values="acc").round(4))

# ---- T2: 위치별 유표성 ----------------------------------------------
mk = tp.stage2_markedness_by_position(paths, X, meta, arrays=ARRAYS_S2,
                                      n_reps=400, tag="", force=True)
if len(mk):
    piv = mk.pivot_table(index=["array", "position"], columns="series", values="d2_cv")
    piv["voiced_minus_voiceless"] = piv["voiced"] - piv["voiceless"]
    print("\n[T2] 위치별 유표성 (교차검증 제곱거리):")
    display(piv.round(5))

# ---- Miller-Nicely 자질 전달량 ---------------------------------------
tr = tp.stage2_feature_transmission(paths, X, meta, arrays=ARRAYS_S2,
                                    positions=("all", "initial", "medial", "final"),
                                    tag="", force=True, max_per_class=300)
if len(tr):
    print("\n[자질 전달량] 혼동행렬에서 계산한 상대 전달정보:")
    display(tr.pivot_table(index=["position", "feature"], columns="array",
                           values="T_rel").round(4))

# ---- T3: 양식 (정렬 품질·날짜 맞춤) ----------------------------------
avail = set(meta["session"])
T3_ARMS = [("primary_0729", VOCAL_PRIMARY, NONVOCAL, None),
           ("primary_0729_no0825", VOCAL_PRIMARY,
            [s for s in NONVOCAL if s != "t12.2022.08.25"], None),
           ("secondary_datematch", VOCAL_DATEMATCH, NONVOCAL, None),
           ("tertiary_allvocal", None, NONVOCAL, None),
           ("primary_0729_initial", VOCAL_PRIMARY, NONVOCAL, "initial")]
t3_frames, t3_gaps = [], []
for label, vocal, nonvocal, positions in T3_ARMS:
    vsel = sorted(avail & set(vocal)) if vocal else None
    nsel = sorted(avail & set(nonvocal))
    if not nsel or (vocal and not vsel):
        print(f"[T3 {label}] 해당 세션 정렬 없음 -> 생략")
        continue
    d = tp.stage2_modality_decode(paths, X, meta, arrays=ARRAYS_S2, n_perm=0,
                                  n_repeats=10, tag=f"_{label}", force=True,
                                  per_table=per_table, max_per_class=300,
                                  partitions="held_out", positions=positions,
                                  session_filter={"vocal": vsel, "nonvocal": nsel})
    if not len(d):
        continue
    d["comparator"] = label
    t3_frames.append(d)
    print(f"\n[T3 {label}] vocal={vsel or 'all held-out'} nonvocal={nsel}")
    display(d.pivot_table(index=["array", "modality"], columns="contrast",
                          values="acc").round(4))
    display(d.groupby("modality")[["n_matched_per_class", "mean_per"]]
            .agg({"n_matched_per_class": "median", "mean_per": "mean"}).round(4))
    gap = tp.contrast_gap_ci(d, group_col="modality", levels=["vocal", "nonvocal"],
                             array="6v", dod_pair=("vocal", "nonvocal"))
    if len(gap):
        gap["comparator"], gap["positions"] = label, (positions or "all")
        t3_gaps.append(gap)
        display(gap)
if t3_frames:
    pd.concat(t3_frames, ignore_index=True).to_csv(
        paths["RESULT_DIR"] / "stage2_t3_comparators.csv", index=False)
if t3_gaps:
    pd.concat(t3_gaps, ignore_index=True).to_csv(
        paths["RESULT_DIR"] / "stage2_t3_gaps.csv", index=False)

# ---- 민감도: 다른 창 / 필터 없음 / 효과를 심은 합성 특징(논리 점검) ----
RUN_SENSITIVITY = True
if RUN_SENSITIVITY:
    alt_align, _ = tp.apply_score_filter(
        tp.load_alignments(paths["ALIGN_DIR"], window_mode="rnn_center"),
        score_min=SCORE_MIN)
    Xa, _, meta_a = tp.stage2_segment_features(paths, alt_align)
    arms = {
        "seg": t1.get("all_pairs_pooled"),
        "rnn_center": tp.stage2_position_decode(
            paths, Xa, meta_a, arrays=("6v",), n_perm=0, tag="_rnncenter",
            force=True, max_per_class=300, match_n_across_positions=True),
    }
    # score 필터 없음
    X_ns, _, meta_ns = tp.stage2_segment_features(paths, align_raw)
    arms["noscore"] = tp.stage2_position_decode(
        paths, X_ns, meta_ns, arrays=("6v",), n_perm=0, tag="_noscore",
        force=True, max_per_class=300, match_n_across_positions=True)
    # 효과를 심은 합성 특징: 분석이 심은 효과를 복원하는지 확인(양성 통제)
    X_syn, _, meta_syn = tp.stage2_synthetic_features(align)
    arms["synthcheck"] = tp.stage2_position_decode(
        paths, X_syn, meta_syn, arrays=("6v",), n_perm=0, tag="_synthcheck",
        force=True, max_per_class=300, match_n_across_positions=True)
    rows = []
    for name, d in arms.items():
        if d is None or not len(d):
            continue
        for position in tp.POSITIONS:
            sp = d[(d["array"] == "6v") & (d["position"] == position)]
            if sp.empty:
                continue
            rows.append({
                "arm": name, "position": position,
                "voicing": round(sp.loc[sp["contrast"] == "laryngeal", "acc"].mean(), 4),
                "place": round(sp.loc[sp["contrast"] == "place", "acc"].mean(), 4),
                "n_median": int(sp["n_per_class"].median())})
    cmp_df = pd.DataFrame(rows)
    cmp_df.to_csv(paths["RESULT_DIR"] / "stage2_arm_comparison.csv", index=False)
    print("\n[민감도] 팔 비교 (area 6v):")
    display(cmp_df)

if t1.get("all_pairs_pooled") is not None and len(t1["all_pairs_pooled"]):
    tp.fig_stage2_position(t1["all_pairs_pooled"],
                           paths["FIG_DIR"] / "stage2_position.png", array="6v")
if len(mk):
    tp.fig_markedness(mk, paths["FIG_DIR"] / "stage2_markedness.png", array="6v")
if len(tr):
    tp.fig_feature_transmission(tr, paths["FIG_DIR"] / "stage2_feature_transmission.png")

stage2 = {"source": "real_alignments", "feature_source": "competitionData",
          "sessions": sorted(meta["session"].unique().tolist()),
          "n_segments": int(len(meta))}

## 추가 분석 (2026-10-09, 외부 검토 반영)

- **PCA 수정**: 분류기 전처리의 PCA·스케일러를 교차검증 학습 폴드 안에서만 적합하도록 `t12_pipeline.py`를 고쳤다. 수정 전 수치는 `results_pre_pcafix/`에 보존.
- **T1′ 파열음 전용 1차 검정**: 후두쌍 P/B·T/D·K/G만, 위치 대조는 파열음 위치쌍, 무기식 대조는 마찰음 후두쌍(F/V·TH/DH). 어두 vs 비어두.
- **자질 정보의 시간 전개**: 고립 음소 과제에서 200 ms 창을 밀며 후두(파열음·마찰음)·조음위치·39음소 정보의 시간 경로를 본다.

아래 두 셀은 `code/` 폴더의 실행 스크립트를 그대로 호출한다(로컬 수십 분). 결과 CSV가 이미 있으면 표만 다시 그린다.

### T1′ 통제 분석 (2026-10-09 오후, 외부 검토 ①②③④)
- **문장 시행 그룹 CV**(`StratifiedGroupKFold`, 블록·세션·단어 그룹 민감도), **발성 세션·held_out 구간만** 주분석, **비어두의 어중:어말 구성 균형화**, **단어 상한 20**, **80 ms 고정 중심창**.
- 통계: 쌍 대응 유지 재표집 CI, 쌍 라벨 정확 순열, **구간 수준 위치 라벨 순열(주 검정)**, 20회 독립 재표집, 쌍 하나씩 제외. 유성·무성 위치쌍 분리 보고.
- 가설·주분석·판정 기준은 실행 전 `results/t1_controls_prereg.md`에 고정.


In [ ]:
# =====================================================================
# 9. T1' 파열음 전용 1차 검정 (어두 vs 비어두)
# =====================================================================
import runpy
from IPython.display import display

RES = paths["RESULT_DIR"]
RERUN_T1_STOPS = False   # True면 재계산(순열 100회, 수십 분)
if RERUN_T1_STOPS or not (RES / "stage2_t1_stops_dod.csv").exists():
    runpy.run_path(str(CODE_DIR / "run_t1_stops_primary.py"), run_name="__main__")

dod = pd.read_csv(RES / "stage2_t1_stops_dod.csv")
cols = ["run", "lar_a", "lar_b", "place_a", "place_b", "gap_a", "gap_a_lo", "gap_a_hi",
        "gap_b", "gap_b_lo", "gap_b_hi", "dod_b_minus_a", "dod_lo", "dod_hi", "n_lar_pairs"]
print("위치(a=어두, b=비어두)별 정확도, 위치−후두 격차, DoD(b−a) [부트스트랩 95% CI]")
display(dod[cols].round(3))
prim = pd.read_csv(RES / "stage2_t1_stops_primary.csv")
for run in prim["run"].unique():
    g = prim[prim["run"] == run]
    print(f"\n[{run}] 쌍별 정확도")
    display(g.pivot_table(index=["contrast", "pair"], columns="position", values="acc").round(3))
print("\n※ 마찰음 대조군(fric_pooled_2lvl)도 같은 크기로 떨어지면 위치 효과는 기식 특이적이 아니라 후두 대립 일반이다.")

In [ ]:
# =====================================================================
# 10. 자질 정보의 시간 전개 (tuningTasks 고립 음소, 6v)
# =====================================================================
import runpy
from IPython.display import Image, display

RERUN_TIMECOURSE = False
if RERUN_TIMECOURSE or not (RES / "stage1_timecourse_summary.csv").exists():
    runpy.run_path(str(CODE_DIR / "run_stage1_timecourse.py"), run_name="__main__")

summ = pd.read_csv(RES / "stage1_timecourse_summary.csv")
print("피크 시점(ms)과 피크 정확도. 지연 과제라 go 전에도 정보가 있어 '잠복기'는 해석하지 않는다.")
display(summ.round(3))
display(Image(filename=str(RES / "figures" / "stage1_timecourse.png")))

In [ ]:
# =====================================================================
# 11. T1′ 통제 분석 (2026-10-09 외부 검토 ①②③④) — 사전 고정 results/t1_controls_prereg.md
#     로컬 병렬 실행은 code/run_t1c_parallel.sh (워커 5개). 여기서는 결과 CSV가 없을 때만 직렬 실행.
# =====================================================================
import runpy
import sys
from IPython.display import display

RES = paths["RESULT_DIR"]
RERUN_T1_CONTROLS = False
if RERUN_T1_CONTROLS or not (RES / "t1c_interaction.csv").exists():
    _argv = sys.argv
    sys.argv = ["run_t1_controls.py", "all", "--perm", "200", "--perm-n", "200", "--n-resample", "20"]
    try:
        runpy.run_path(str(CODE_DIR / "run_t1_controls.py"), run_name="__main__")
    finally:
        sys.argv = _argv

MAIN = "main_heldout_trialcv_bal"
it = pd.read_csv(RES / "t1c_interaction.csv")
m = it[(it["lar_cat"] == "stops_lar") & (it["place_cat"] == "place_all")]
print("D = Δ_place − Δ_lar, Δ = acc(비어두) − acc(어두). 쌍 대응 유지 부트스트랩 95% CI, 쌍 라벨 정확 순열 p(최소 .029)")
display(m[["run", "pos_a", "pos_b", "delta_lar", "delta_place", "D", "D_lo", "D_hi", "p_label_one"]].round(3))
print("\n주분석 구간 수준 위치 라벨 순열(쌍 수와 무관한 주 검정):")
print((RES / "t1c_position_perm_summary.txt").read_text())
sub = pd.read_csv(RES / "t1c_voicing_subgroups.csv")
print("\n주분석 범주별: 후두 파열 / 후두 마찰 / 무성 위치 / 유성 위치 (acc_a=어두, acc_b=비어두)")
display(sub[sub["run"] == MAIN][["category", "n_pairs", "acc_a", "acc_b", "delta", "pairs"]].round(3))
loo = pd.read_csv(RES / "t1c_leave_one_pair_out.csv")
print("\n주분석 쌍 하나씩 제외한 D")
display(loo[loo["run"] == MAIN][["dropped", "D", "D_lo", "D_hi"]].round(3))
rs = pd.read_csv(RES / "t1c_resample_summary.csv")
print("\n주분석 20회 독립 재표집: 반복 간 D 분포")
display(rs[["lar_cat", "place_cat", "D_mean_over_reps", "D_rep_p2_5", "D_rep_p97_5", "frac_reps_D_pos", "p_label_one"]].round(3))


In [ ]:
# =====================================================================
# 8. 요약 — 압축 결과표 출력 + results/SUMMARY.md 작성
# =====================================================================

summary_path = tp.write_summary(paths, extra={
    "LOCAL": LOCAL,
    "stage2_alignment_source": stage2["source"],
    "stage2_feature_source": stage2["feature_source"],
    "stage2_sessions": stage2["sessions"],
    "caveat": "음성 자질표 verified=False. 사전등록 전 확정 금지.",
})

print()
print(summary_path.read_text(encoding="utf-8")[:4000])